In [ ]:
import os, glob, sqlite3
import pandas as pd
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive')
# --- 1) Config: Everyone uses this same config ---

# This is the name of the folder you shared and they created a shortcut to
SHARED_FOLDER_NAME = "Project_Data"
DATASET_NAME = "dataset.csv"
DB_NAME = "dataset.db"

# Build the full, shared paths
# This path will work for EVERYONE, *if* they followed the instructions
# to add the shortcut to their "My Drive"
BASE_PATH = os.path.join("/content/drive/MyDrive", SHARED_FOLDER_NAME)
CSV_PATH = os.path.join(BASE_PATH, DATASET_NAME)
DB_PATH = os.path.join(BASE_PATH, DB_NAME)

TABLE_NAME = "dataset"

Mounted at /content/drive


In [ ]:
# --- 2) Utility: find the CSV in your Drive (MyDrive); searches recursively if needed ---
def find_csv_in_mydrive(filename=DATASET_NAME):
    # Common direct location
    direct = f"/content/drive/MyDrive/{filename}"
    if os.path.exists(direct):
        return direct
    # Recursive search (can take a bit if you have a lot of files)
    matches = glob.glob(f"/content/drive/MyDrive/**/{filename}", recursive=True)
    if matches:
        return matches[0]
    raise FileNotFoundError(
        f"Could not find {filename} under /content/drive/MyDrive. "
        f"Try moving it there or update DATASET_NAME/paths."
    )


In [ ]:
# --- 3) Load CSV into SQLite helper ---
def load_csv_to_sqlite(csv_path, db_path=DB_PATH, table_name=TABLE_NAME, chunksize=100_000):
    """
    Loads a CSV into SQLite using pandas.to_sql.
    - Uses chunking for large files.
    - Replaces the table on the first chunk, then appends.
    """
    con = sqlite3.connect(db_path)
    try:
        first = True
        for chunk in pd.read_csv(csv_path, chunksize=chunksize):
            chunk.to_sql(table_name, con, if_exists="replace" if first else "append", index=False)
            first = False
        con.commit()
    finally:
        con.close()

In [ ]:
def query_db(sql, params=None, db_path=DB_PATH):
    """
    Run a SELECT (or PRAGMA) and return a pandas DataFrame.
    """
    con = sqlite3.connect(db_path)
    try:
        df = pd.read_sql_query(sql, con, params=params)
        return df
    finally:
        con.close()

In [ ]:
csv_path = find_csv_in_mydrive(DATASET_NAME)
print(f"Using CSV: {csv_path}")

# Create/overwrite DB & load data
load_csv_to_sqlite(csv_path, db_path=DB_PATH, table_name=TABLE_NAME)
print(f"Loaded data into {DB_PATH} (table: {TABLE_NAME})")

# Show schema (column list) and a sample query
schema_df = query_db(f"PRAGMA table_info({TABLE_NAME});")
print("\nTable schema:")
display(schema_df)

# Below code puts the ALL data from the csv into a pandas dataframe
df = query_db(f"SELECT * FROM {TABLE_NAME};")
print("\nSample rows:")
display(df.head(10))

Using CSV: /content/drive/MyDrive/dataset.csv
Loaded data into /content/drive/MyDrive/dataset.db (table: dataset)

Table schema:


,cid,name,type,notnull,dflt_value,pk
0,0,feeling.nervous,TEXT,0,None,0
1,1,panic,TEXT,0,None,0
2,2,breathing.rapidly,TEXT,0,None,0
3,3,sweating,TEXT,0,None,0
4,4,trouble.in.concentration,TEXT,0,None,0
5,5,having.trouble.in.sleeping,TEXT,0,None,0
6,6,having.trouble.with.work,TEXT,0,None,0
7,7,hopelessness,TEXT,0,None,0
8,8,anger,TEXT,0,None,0
9,9,over.react,TEXT,0,None,0



Sample rows:


,feeling.nervous,panic,breathing.rapidly,sweating,trouble.in.concentration,having.trouble.in.sleeping,having.trouble.with.work,hopelessness,anger,over.react,...,weight.gain,material.possessions,introvert,popping.up.stressful.memory,having.nightmares,avoids.people.or.activities,feeling.negative,trouble.concentrating,blamming.yourself,Disorder
0,yes,yes,yes,yes,yes,yes,no,no,no,no,...,no,no,no,no,no,no,no,no,no,Anxiety
1,no,no,no,no,no,no,yes,yes,yes,yes,...,no,no,no,no,no,no,no,no,no,Depression
2,no,no,no,no,no,no,no,no,no,no,...,yes,yes,yes,no,no,no,no,no,no,Loneliness
3,no,no,no,no,no,no,no,no,no,no,...,no,no,no,yes,yes,yes,yes,yes,yes,Stress
4,no,no,no,no,no,no,no,no,no,no,...,no,no,no,no,no,no,no,no,no,Normal
5,yes,yes,yes,yes,yes,yes,no,no,no,no,...,no,no,no,no,no,no,no,no,no,Anxiety
6,no,no,no,no,no,no,yes,yes,yes,yes,...,no,no,no,no,no,no,no,no,no,Depression
7,no,no,no,no,no,no,no,no,no,no,...,yes,yes,yes,no,no,no,no,no,no,Loneliness
8,no,no,no,no,no,no,no,no,no,no,...,no,no,no,yes,yes,yes,yes,yes,yes,Stress
9,no,no,no,no,no,no,no,no,no,no,...,no,no,no,no,no,no,no,no,no,Normal


In [ ]:
sample_df = query_db(f"SELECT disorder, count(disorder) as num_of_records_with_disorder FROM {TABLE_NAME} group by disorder;")
print("\nSample rows:")
display(sample_df)


Sample rows:


,Disorder,num_of_records_with_disorder
0,Anxiety,8192
1,Depression,8192
2,Loneliness,8192
3,Normal,8192
4,Stress,8192
